In [ ]:
%%writefile device_query.cu
#include <stdio.h>
#include <stdlib.h>

int main()
{
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);
    if (deviceCount == 0)
    {
        printf("There is no device supporting CUDA\n");
    }
    int dev;
    for (dev = 0; dev < deviceCount; ++dev)
    {
        cudaDeviceProp deviceProp;
        cudaGetDeviceProperties(&deviceProp, dev);
        if (dev == 0)
        {
            if (deviceProp.major < 1)
            {
                printf("There is no device supporting CUDA.\n");
            }
            else if (deviceCount == 1)
            {
                printf("There is 1 device supporting CUDA\n");
            }
            else
            {
                printf("There are %d devices supporting CUDA\n", deviceCount);
            }
        }
        printf("\nDevice %d: \"%s\"\n", dev, deviceProp.name);
        printf("  Major revision number:                         %d\n", deviceProp.major);
        printf("  Minor revision number:                         %d\n", deviceProp.minor);
        printf("  Total amount of global memory:                 %zu bytes\n", deviceProp.totalGlobalMem);
        printf("  Total amount of constant memory:               %zu bytes\n", deviceProp.totalConstMem);
        printf("  Total amount of shared memory per block:       %zu bytes\n", deviceProp.sharedMemPerBlock);
        printf("  Total number of registers available per block: %d\n", deviceProp.regsPerBlock);
        printf("  Warp size:                                     %d\n", deviceProp.warpSize);
        printf("  Multiprocessor count:                          %d\n",deviceProp.multiProcessorCount );

        printf("  Maximum number of threads per block:           %d\n", deviceProp.maxThreadsPerBlock);
        printf("  Maximum sizes of each dimension of a block:    %d x %d x %d\n", deviceProp.maxThreadsDim[0],deviceProp.maxThreadsDim[1], deviceProp.maxThreadsDim[2]);
        printf("  Maximum sizes of each dimension of a grid:     %d x %d x %d\n", deviceProp.maxGridSize[0], deviceProp.maxGridSize[1],  deviceProp.maxGridSize[2]);
        printf("  Maximum memory pitch:                          %zu bytes\n", deviceProp.memPitch);
        printf("  Texture alignment:                             %zu bytes\n", deviceProp.textureAlignment);
        printf("  Clock rate:                                    %d kilohertz\n", deviceProp.clockRate);
    }
}


Overwriting device_query.cu


In [ ]:
!nvcc device_query.cu -o device_query

In [ ]:
!./device_query


There is 1 device supporting CUDA

Device 0: "Tesla T4"
  Major revision number:                         7
  Minor revision number:                         5
  Total amount of global memory:                 15835660288 bytes
  Total amount of constant memory:               65536 bytes
  Total amount of shared memory per block:       49152 bytes
  Total number of registers available per block: 65536
  Warp size:                                     32
  Multiprocessor count:                          40
  Maximum number of threads per block:           1024
  Maximum sizes of each dimension of a block:    1024 x 1024 x 64
  Maximum sizes of each dimension of a grid:     2147483647 x 65535 x 65535
  Maximum memory pitch:                          2147483647 bytes
  Texture alignment:                             512 bytes
  Clock rate:                                    1590000 kilohertz


In [ ]:
%%writefile hello_world_threads.cu
#include <stdio.h>

__global__ void helloWorldFromGPU()
{
    int threadID = threadIdx.x;
    printf("Hello World from thread ID: %d\n", threadID);
}

int main()
{
    int threadsPerBlock = 10; // Number of threads per block
    helloWorldFromGPU<<<1, threadsPerBlock>>>();
    cudaDeviceSynchronize();
    return 0;
}


Writing hello_world_threads.cu


In [ ]:
!nvcc hello_world_threads.cu -o hello_world_threads

In [ ]:
!./hello_world_threads

Hello World from thread ID: 0
Hello World from thread ID: 1
Hello World from thread ID: 2
Hello World from thread ID: 3
Hello World from thread ID: 4
Hello World from thread ID: 5
Hello World from thread ID: 6
Hello World from thread ID: 7
Hello World from thread ID: 8
Hello World from thread ID: 9


In [ ]:
%%writefile hello_world_multiple_blocks.cu
#include <stdio.h>

__global__ void helloWorldFromGPU()
{
    int threadID = threadIdx.x;
    int blockID = blockIdx.x;
    int globalThreadID = blockID * blockDim.x + threadID;
    printf("Hello World from block %d, thread %d (global thread ID: %d)\n", blockID, threadID, globalThreadID);
}

int main()
{
    int threadsPerBlock = 5;
    int numberOfBlocks = 3;
    helloWorldFromGPU<<<numberOfBlocks, threadsPerBlock>>>();
    cudaDeviceSynchronize();
    return 0;
}


Writing hello_world_multiple_blocks.cu


In [ ]:
!nvcc hello_world_multiple_blocks.cu -o hello_world_multiple_blocks

In [ ]:
!./hello_world_multiple_blocks

Hello World from block 2, thread 0 (global thread ID: 10)
Hello World from block 2, thread 1 (global thread ID: 11)
Hello World from block 2, thread 2 (global thread ID: 12)
Hello World from block 2, thread 3 (global thread ID: 13)
Hello World from block 2, thread 4 (global thread ID: 14)
Hello World from block 0, thread 0 (global thread ID: 0)
Hello World from block 0, thread 1 (global thread ID: 1)
Hello World from block 0, thread 2 (global thread ID: 2)
Hello World from block 0, thread 3 (global thread ID: 3)
Hello World from block 0, thread 4 (global thread ID: 4)
Hello World from block 1, thread 0 (global thread ID: 5)
Hello World from block 1, thread 1 (global thread ID: 6)
Hello World from block 1, thread 2 (global thread ID: 7)
Hello World from block 1, thread 3 (global thread ID: 8)
Hello World from block 1, thread 4 (global thread ID: 9)


In [ ]:
%%writefile hello_world_2D_blocks_threads.cu
#include <stdio.h>

__global__ void helloWorldFromGPU()
{
    int threadIDx = threadIdx.x;
    int threadIDy = threadIdx.y;
    int blockIDx = blockIdx.x;
    int blockIDy = blockIdx.y;
    int globalThreadIDx = blockIDx * blockDim.x + threadIDx;
    int globalThreadIDy = blockIDy * blockDim.y + threadIDy;
    printf("Hello World from block (%d, %d), thread (%d, %d) (global thread ID: (%d, %d))\n",
           blockIDx, blockIDy, threadIDx, threadIDy, globalThreadIDx, globalThreadIDy);
}

int main()
{
    dim3 threadsPerBlock(3, 3);  // 2D threads (3x3)
    dim3 numberOfBlocks(2, 2);   // 2D blocks (2x2)
    helloWorldFromGPU<<<numberOfBlocks, threadsPerBlock>>>();
    cudaDeviceSynchronize();
    return 0;
}


Writing hello_world_2D_blocks_threads.cu


In [ ]:
!nvcc hello_world_2D_blocks_threads.cu -o hello_world_2D_blocks_threads

In [ ]:
!./hello_world_2D_blocks_threads

Hello World from block (0, 1), thread (0, 0) (global thread ID: (0, 3))
Hello World from block (0, 1), thread (1, 0) (global thread ID: (1, 3))
Hello World from block (0, 1), thread (2, 0) (global thread ID: (2, 3))
Hello World from block (0, 1), thread (0, 1) (global thread ID: (0, 4))
Hello World from block (0, 1), thread (1, 1) (global thread ID: (1, 4))
Hello World from block (0, 1), thread (2, 1) (global thread ID: (2, 4))
Hello World from block (0, 1), thread (0, 2) (global thread ID: (0, 5))
Hello World from block (0, 1), thread (1, 2) (global thread ID: (1, 5))
Hello World from block (0, 1), thread (2, 2) (global thread ID: (2, 5))
Hello World from block (0, 0), thread (0, 0) (global thread ID: (0, 0))
Hello World from block (0, 0), thread (1, 0) (global thread ID: (1, 0))
Hello World from block (0, 0), thread (2, 0) (global thread ID: (2, 0))
Hello World from block (0, 0), thread (0, 1) (global thread ID: (0, 1))
Hello World from block (0, 0), thread (1, 1) (global thread ID: 